# Controlled Generative UI
So far our agent has only replied in text. Now it can show React UI: flight cards, pie charts, anything you wire up. You build the components; the agent picks which to use and when.

## Objectives
1. **Understand Generative UI** - What it is and where Controlled Generative UI sits on the spectrum
2. **Register frontend components** - Use `useComponent()` to expose React components to the agent
3. **Render structured tool output** - Let the agent choose and populate UI components in chat

## What are we building

A chat interface that renders rich UI components — flight cards, pie charts — based on what the user asks for:

Show a flight card for Pacific Air from SFO to JFK departing at 08:30 for $249

Please show me the distribution of our revenue by category in a pie chart

## What is Controlled Generative UI?
Generative UI is a pattern where agents respond with fully interactive interfaces, not just text. **Controlled Generative UI** is the most constrained variant: the agent can only render components you explicitly register.

### How it works
Each registered component is exposed as a tool with:
- a stable name
- a typed input schema
- a mapped React component

The agent doesn't generate arbitrary UI. It passes structured data into components you've built, all defined on the frontend and registered at runtime.

### Pros and cons
**Pros**
- Easy to implement: register a component and you're done.
- High visual polish, since every rendered surface is one you authored.
- Strong safety: the model can only call registered tools with validated arguments.
- Good fit for high-traffic or mission-critical UX where stability matters.

**Cons**
- Frontend effort grows with every new capability: each pattern needs its own component.
- Less expressive freedom than declarative or open-ended generative UI.

### The `useComponent()` hook
[useComponent](https://docs.copilotkit.ai/generative-ui/your-components/display-only) registers a React component as a tool the agent can call inside `<CopilotChat />`. You define what's available; the agent picks when to use it.

```tsx
useComponent({
  name: "component_name",
  description: "description for the agent to know about this component",
  parameters: z.object({ ... }),
  render: MyComponent,
});
```

- `name` (required `string`): tool name exposed to the model.
- `description` (optional `string`): tells the model when to call this tool.
- `parameters` (optional Zod schema): structured props passed in as arguments.
- `render` (required): a React component (rendered as `<Component {...args} />`), or a function receiving `{ args, status }` for custom rendering (loading states, wrappers, conditional display).

## Setup Dependencies

In [2]:
# install dependencies
# !pip install -r requirements.txt

In [3]:
from helper import load_api_keys
load_api_keys()

✓ OpenAI API key loaded
✓ Google API key loaded


## Start Agent

In [ ]:
# Start the same agent backend:
from backend.server import start_backend
start_backend(port=8003)

✓ Server running at http://localhost:8003


## Start Frontend

In [5]:
# start front end
from helper import start_frontend
start_frontend(port=3003)

Installing frontend dependencies ...
npm warn deprecated lodash.get@4.4.2: This package is deprecated. Use the optional chaining (?.) operator instead.
npm warn deprecated hast@1.0.0: Renamed to rehype
npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead

added 970 packages, and audited 971 packages in 13s

249 packages are looking for funding
  run `npm fund` for details

15 vulnerabilities (14 moderate, 1 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
npm notice
npm notice New major version of npm available! 10.9.7 -> 11.14.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.14.1
npm notice To update run: npm install -g npm@11.14.1
npm notice
✓ Frontend dependencies installed
Starting frontend on port 3003 ...
✓ App running at https://s172-29-31-106p3003.lab-aws-production.deeplearning.ai


In [6]:
# preview the app
from helper import display_app
display_app(port=3003)

## Register components

The example below registers three components with `useComponent()`: a simple `showName` card, a pie chart, and a flight card.

In [7]:
%%writefile frontend/src/App.tsx
import { z } from "zod"
import { CopilotChat } from "@copilotkit/react-core/v2";
import { useComponent } from "@copilotkit/react-core/v2";

import { FlightCard, FlightCardProps } from "@/components/flight-card";
import { PieChart, PieChartProps } from "@/components/pie-chart";

import { useExampleSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {

  // 🪁 Register a component that shows the name of the user
  useComponent({
    name: "showMyName",
    description: "Show the user's name in a card",
    parameters: z.object({ name: z.string() }),
    render: ({ name }) => <div className="bg-blue-500 p-4">Hi, {name}!</div>,
  });

  // 🪁 Resgister a pieChart component to show structured data
  useComponent({
    name: "pieChart",
    description: "Controlled Generative UI that displays data as a pie chart.",
    parameters: PieChartProps,
    render: PieChart,
  });

  // 🪁 Resgister a flightCard component to show flight data
  useComponent({
    name: "flightCard",
    description: "Controlled Generative UI that displays a single flight summary card.",
    parameters: FlightCardProps,
    render: FlightCard,
  });

  // 🪁 Add suggestions to our CopilotChat, will display through buttons
  useExampleSuggestions();

  return <CopilotChat />;

};

Overwriting frontend/src/App.tsx


## Display the app

Your agent now has three components it can render. Try prompting it to use each one.
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Show my name</div>
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Pie chart</div>
<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Flight card</div>


In [ ]:
from helper import display_app
display_app(port=3003)

### Recap

Any React component can become a generative UI tool just register it with `useComponent()`. Here's the full `FlightCard` for reference:
```tsx
import { z } from "zod";

export const FlightCardProps = z.object({
  title: z.string().describe("Flight card title"),
  airline: z.string().describe("Airline name"),
  origin: z.string().describe("Departure airport/city"),
  destination: z.string().describe("Arrival airport/city"),
  departure_time: z.string().describe("Departure time"),
  price: z.string().describe("Price display"),
});

type FlightCardProps = z.infer<typeof FlightCardProps>;

export function FlightCard({
  title,
  airline,
  origin,
  destination,
  departure_time,
  price,
}: FlightCardProps) {
  return (
    <div className="rounded-lg border bg-white p-3 space-y-2">
      <div className="font-semibold">{title}</div>
      <div className="rounded border p-2 text-sm">
        <div className="font-medium">{airline}</div>
        <div>
          {origin} → {destination}
        </div>
        <div>Departs: {departure_time}</div>
        <div className="font-semibold">{price}</div>
      </div>
    </div>
  );
}
```

**Why Zod?**

[Zod](https://zod.dev) is a TypeScript-first schema and validation library — similar to [Pydantic](https://docs.pydantic.dev) in Python. It lets you define the component props once and use the same schema for both the TypeScript type definition and the `parameters` passed to `useComponent()`.